# Extrator Renda Fixa - Banco Inter
Notebook separado em blocos para testar cada etapa sem precisar reabrir o navegador.

In [1]:
# === CÉLULA 1: Imports ===
from seleniumwire import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from selenium.common.exceptions import TimeoutException
import json
import requests

URL_RENDA_FIXA = "rendas-fixas/produtos"
ARQUIVO_SAIDA = "resultado_renda_fixa.json"

print("Imports carregados!")

Imports carregados!


In [2]:
# === CÉLULA 2: Abrir navegador e ir para a página de login ===
# Rode esta célula apenas UMA VEZ. Depois faça login manualmente no navegador.

options = webdriver.ChromeOptions()
options.add_experimental_option("detach", True)
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 30)

driver.get("https://contadigital.inter.co/investimento/renda-fixa")
print("Navegador aberto! Faça login manualmente e depois rode a próxima célula.")

Navegador aberto! Faça login manualmente e depois rode a próxima célula.


In [3]:
# === CÉLULA 3: Aguardar login ===
# Rode esta célula DEPOIS de fazer login no navegador.

try:
    wait.until(EC.presence_of_element_located((By.XPATH, "//span[contains(text(), 'Renda')]")))
    print("Login detectado com sucesso!")
except TimeoutException:
    print("Tempo esgotado. O login não foi concluído. Rode esta célula novamente após logar.")

Login detectado com sucesso!


In [4]:
# === CÉLULA 4: Navegar para Renda Fixa e capturar tokens ===
# Pode re-rodar esta célula sem precisar reabrir o navegador.

driver.requests.clear()  # limpa o histórico pra não pegar lixo de antes

print("\nCapturando tokens...")
try:
    req = driver.wait_for_request(URL_RENDA_FIXA, timeout=10)
except TimeoutException:
    print("Não encontrei a chamada de rendas-fixas/produtos a tempo.")
    raise

auth_header = req.headers.get("authorization")
mag_id = req.headers.get("mag-identifier")

if not auth_header or not mag_id:
    print("A requisição apareceu, mas sem os headers esperados.")
    raise RuntimeError("Headers não encontrados")

print("Headers capturados com sucesso!")
print(f"  auth_header: {auth_header[:30]}...")
print(f"  mag_id: {mag_id}")


Capturando tokens...
Headers capturados com sucesso!
  auth_header: Bearer 176ae71a-5054-456d-865a...
  mag_id: L1plbDc0MXl4UXBmZFFSZHRldGdLZGdXeVNBPQ==


In [13]:
# === CÉLULA 5: Fetch dos dados via JS no contexto do navegador ===
# Re-rode esta célula para refazer a requisição sem navegar de novo.

url_api = "https://cd.web.bancointer.com.br/ib-pfj/investimentos/v2/rendas-fixas/produtos"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "authorization": auth_header,
    "mag-identifier": mag_id
}

resultado = requests.get(url=url_api, headers=headers)
resultado.raise_for_status

data = resultado.json()

print(f"Total de produtos: {len(data) if isinstance(data, list) else 'N/A'}")

Total de produtos: 462


In [14]:
# === CÉLULA 6: Processar e salvar resultados ===

# Salva JSON completo
with open(ARQUIVO_SAIDA, "w", encoding="utf-8") as arquivo:
    json.dump(data, arquivo, ensure_ascii=False, indent=4)
print(f"JSON salvo em {ARQUIVO_SAIDA}")

# Salva TXT filtrado
count = 0
with open("investimentos.txt", "w", encoding="utf-8") as arquivo:
    for investimento in data:
        if investimento.get("grauRisco") in [1, 2, 3] and investimento.get("tipo", {}).get("descricao") in [ "LCI", "LCA", "CDB"]:
            nome = investimento.get("nome", "")
            taxa = investimento.get("taxa", 0)
            inv_min = investimento.get("aplicacaoMinima", 0)
            isento_ir = investimento.get("isentoImpostos", None)
            vencimento = "/".join(investimento.get("dataResgate", "").split("-")[::-1])

            arquivo.write(f"{nome} | Taxa: {taxa} | Inv.Mínimo: {inv_min} | Isento de Imposto de Renda: {isento_ir} | Vencimento: {vencimento}\n")
            count += 1
print(f"TXT salvo em investimentos.txt ({count} investimentos filtrados)")


JSON salvo em resultado_renda_fixa.json
TXT salvo em investimentos.txt (110 investimentos filtrados)
